# Konu 4 — Temsil ve gömme vektörleri

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

## Isınma — elle iki eksenli harita

Konu 3'te bir renk üç sayıydı. Şimdi kelimeleri sayıya çevireceğiz; önce elle.
Bir mood-board düşün, iki ekseni var:

- **sıcaklık:** soğuk −1 … sıcak +1
- **enerji:** sakin −1 … canlı +1

Altı kelimeye ikişer sayı verdik: ilki sıcaklık, ikincisi enerji.

In [ ]:
kahve = [0.6, 0.4]
cay = [0.5, -0.4]
buz = [-0.9, 0.1]
deniz = [-0.5, -0.5]
ates = [0.9, 0.9]
kar = [-0.7, -0.2]

print(cay)

Çizerken her noktanın yanına adını yazacağız. Bunun için bir sözlük: anahtar kelimenin adı, değer iki sayısı.

In [ ]:
harita = {
    "kahve": kahve,
    "çay": cay,
    "buz": buz,
    "deniz": deniz,
    "ateş": ates,
    "kar": kar,
}
print(harita["ateş"])

Haritayı çizelim. Her kelime bir nokta; `plt.text` noktanın yanına kelimeyi yazar.
Gri çizgiler sıfır çizgileri: sağ taraf sıcak, üst taraf canlı.

In [ ]:
import matplotlib.pyplot as plt

for kelime in harita:
    nokta = harita[kelime]
    plt.scatter(nokta[0], nokta[1], color="#4a6fa5")
    plt.text(nokta[0], nokta[1], kelime)

plt.axhline(0, color="lightgray")
plt.axvline(0, color="lightgray")
plt.xlabel("soğuk — sıcak")
plt.ylabel("sakin — canlı")
plt.show()

Kelimeyi sayıya çevirdik; yakın duranlar birbirine benziyor. Eksenleri **biz** seçtik.
Model de böyle bir harita kurar, ama eksenleri kendisi bulur: 2 değil, yüzlerce.

İki kelimenin ne kadar benzediğini tek sayıyla söyleyen **hazır bir araç** var:
`cosine_similarity` (Konu 3'teki scikit-learn'den). Onu küçük bir fonksiyona koyuyoruz.
İçini bilmen gerekmiyor; verdiği sayıyı okuman yeter:

- **1**: aynı yön · **0**: ilgisiz · **−1**: zıt yön

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def benzerlik(a, b):
    sonuc = cosine_similarity([a], [b])
    satir = sonuc[0]
    return satir[0]

In [ ]:
print(benzerlik(kahve, cay))
print(benzerlik(kahve, buz))

Kahve ile çay ikisi de sıcak tarafta: yaklaşık 0.3. Buz karşı tarafta: yaklaşık −0.77.

Hücrede `buz` yerine `ates` yaz: 0.98. Ateş kahveden çok daha sıcak ve canlı, ama **aynı yönde**.
Benzerlik, ortadaki sıfır noktasından bakınca iki kelimenin aynı yöne düşüp düşmediğine bakar.

## Adım 1 — Anahtarı oku

Konu 2'deki hücrenin aynısı. `anahtar.txt` bir üst klasörde, `gita3111`'in içinde durur.

In [ ]:
anahtarlar = {}
with open("../anahtar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        parcalar = satir.split("=")
        ad = parcalar[0].strip()
        deger = parcalar[1].strip()
        anahtarlar[ad] = deger

hesap = anahtarlar["ACCOUNT_ID"]
anahtar = anahtarlar["API_TOKEN"]
print("Anahtar okundu")

## Adım 2 — Bir kelimenin vektörü

Konu 2'deki isteğin aynısı, iki farkla:

- **Model** başka: metin yazan değil, metni sayılara çeviren bir model (Google'ın, 100'den çok dil biliyor).
- **Gövde** başka: `"text"` ve bir liste.

In [ ]:
import requests

MODEL = "@cf/google/embeddinggemma-300m"
adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{MODEL}"
basliklar = {"Authorization": f"Bearer {anahtar}"}
govde = {"text": ["kahve"]}

In [ ]:
cevap = requests.post(adres, headers=basliklar, json=govde)
print(cevap.status_code)

Yanıta katman katman iniyoruz. `data` bir liste, çünkü tek istekte birden çok metin
gönderebiliriz. Biz tek kelime gönderdik; ilki bizim.

In [ ]:
yanit = cevap.json()
sonuc = yanit["result"]
vektorler = sonuc["data"]
vektor = vektorler[0]

In [ ]:
print(len(vektor))
print(vektor[:5])

Isınmada "kahve" 2 sayıydı. Model kaç sayı verdi? Bu sayı listesine **vektör**,
bu çeviriye **gömme** (embedding) diyoruz. Sayıların tek tek anlamı yok; anlam,
kelimelerin birbirine göre nerede durduğunda.

## Adım 3 — Fonksiyona koy

Konu 2'deki `modele_sor`'un kardeşi: kelimeyi alır, vektörünü geri verir.
İçindeki satırlar Adım 2'nin aynısı.

In [ ]:
def vektor_al(kelime):
    govde = {"text": [kelime]}
    cevap = requests.post(adres, headers=basliklar, json=govde)
    yanit = cevap.json()
    sonuc = yanit["result"]
    vektorler = sonuc["data"]
    return vektorler[0]

In [ ]:
vektor = vektor_al("çay")
print(len(vektor))

## Adım 4 — İki kelime ne kadar benzer?

**Çalıştırmadan önce tahmin et:** kahve hangisine daha çok benzer, çaya mı tipografiye mi?

Isınmadaki `kahve` 2 sayıydı; şimdi aynı ad modelin verdiği vektörü tutuyor.
`benzerlik` fonksiyonu ikisiyle de çalışır. (`benzerlik` ısınmada tanımlandı; ısınmayı
atladıysan `NameError` alırsın.)

In [ ]:
kahve = vektor_al("kahve")
cay = vektor_al("çay")
tipografi = vektor_al("tipografi")

In [ ]:
print(benzerlik(kahve, cay))
print(benzerlik(kahve, tipografi))

Hangisi büyük çıktı? Tahminin tuttu mu? Bu sayılara tek başına değil, **yan yana** bak:
hangisi daha büyük?

## Adım 5 — 24 kelimelik sözlük

Dört grup, altışar kelime: renkler, duygular, tasarım, mekânlar.

In [ ]:
kelimeler = [
    "kırmızı", "mavi", "yeşil", "sarı", "siyah", "beyaz",
    "huzur", "öfke", "neşe", "hüzün", "heyecan", "sakinlik",
    "tipografi", "logo", "afiş", "palet", "kontrast", "serif",
    "kafe", "kütüphane", "park", "ofis", "atölye", "sahne",
]
print(len(kelimeler))

Her kelime için ayrı bir istek gidiyor: 24 istek, hücre biraz sürer.
Sözlükte her kelimenin karşısına onun vektörü yazılıyor.

In [ ]:
vektorler_sozlugu = {}
for kelime in kelimeler:
    vektorler_sozlugu[kelime] = vektor_al(kelime)

print(len(vektorler_sozlugu))

## Adım 6 — "kafe"ye en yakın 3 kelime

Bu adımı **sen yaz**. Plan:

1. `skorlar` adında boş bir sözlük kur.
2. Sözlükteki her kelime için "kafe" ile o kelimenin benzerliğini hesapla, `skorlar[kelime]`'ye koy.
3. Konu 1'deki `Counter` ile en büyükleri sırala.

Önce kendin dene, sonra aşağıdaki hücrelerle karşılaştır.

In [ ]:
hedef = "kafe"
hedef_vektor = vektorler_sozlugu[hedef]

skorlar = {}
for kelime in vektorler_sozlugu:
    vektor = vektorler_sozlugu[kelime]
    skorlar[kelime] = benzerlik(hedef_vektor, vektor)

print(len(skorlar))

Konu 1'in `Counter`'ı geri döndü. Orada adetleri sıralıyordu, burada benzerlikleri.

In [ ]:
from collections import Counter

sirali = Counter(skorlar)
for kelime, skor in sirali.most_common(4):
    print(kelime, round(skor, 2))

**3 komşu istiyorduk, neden 4 yazdık?** İlk satıra bak.

Konu 1'de müşteriler bu kafeyi **sessiz bir çalışma yeri** diye anlatmıştı.
Model "kafe"yi neye yakın buluyor: kütüphaneye mi, parka mı, sahneye mi?

`hedef = "kafe"` yerine `"huzur"` ya da `"afiş"` yaz, iki hücreyi yeniden çalıştır.

## Adım 7 — Harita

Isınmadaki gibi bir harita istiyoruz. Ama her kelimede yüzlerce sayı var, kâğıt ise iki boyutlu.
**PCA** (scikit-learn'den) yüzlerce sayıyı, en çok şey anlatan 2 sayıya indirir.

Önce bütün vektörleri bir listeye koyuyoruz:

In [ ]:
liste = []
for kelime in vektorler_sozlugu:
    liste.append(vektorler_sozlugu[kelime])

print(len(liste))

`fit`: Konu 3'teki gibi "önce örneklere bak". PCA bütün vektörlere bakıp en çok şey anlatan iki yönü bulur.
Hücrenin altında `PCA(n_components=2)` yazan bir kutu çıkar: hazır demek.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(liste)

Şimdi her kelimeyi 2 sayıya çeviriyoruz ve ısınmadaki gibi bir sözlüğe koyuyoruz.
`transform`, Konu 3'teki `predict` gibi bir liste ister: `[vektor]`. Cevap da liste içinde gelir: `noktalar[0]`.

In [ ]:
model_haritasi = {}
for kelime in vektorler_sozlugu:
    vektor = vektorler_sozlugu[kelime]
    noktalar = pca.transform([vektor])
    model_haritasi[kelime] = noktalar[0]

print(model_haritasi["kafe"])

"kafe" artık ısınmadaki gibi 2 sayı. Çizim, ısınmadaki döngünün aynısı; yalnızca sözlüğün adı değişti.

In [ ]:
for kelime in model_haritasi:
    nokta = model_haritasi[kelime]
    plt.scatter(nokta[0], nokta[1], color="#4a6fa5")
    plt.text(nokta[0], nokta[1], kelime)

plt.show()

Yüzlerce sayıyı 2'ye indirdik. Harita bir **özet**: bilgi kaybeder.
Haritada yan yana duran iki kelime, gerçekte o kadar yakın olmayabilir.

## Adım 8 — Haritayı oku

- Dört grup (renkler, duygular, tasarım, mekânlar) ayrı kümeler oluşturdu mu?
- Kendi grubundan kaçan bir kelime var mı? Nereye gitmiş?
- **huzur** ile **öfke** zıt anlamlı. Haritada uzak mı düştüler?

Haritaya güvenmeden önce asıl sayılara bakalım:

In [ ]:
huzur = vektorler_sozlugu["huzur"]
ofke = vektorler_sozlugu["öfke"]
sakinlik = vektorler_sozlugu["sakinlik"]

print(benzerlik(huzur, ofke))
print(benzerlik(huzur, sakinlik))

Zıt anlamlı iki kelime de yakın düşebilir: ikisi de duygulardan söz eden cümlelerde,
aynı bağlamda geçer. Modelin "benzer"i **eşanlamlı** demek değil, **benzer yerlerde geçen** demek.

## Bonus — Kendi 5 kelimen

Listeye kendi beş kelimeni yaz, çalıştır. Sonra Adım 7'nin dört hücresini sırayla yeniden çalıştır.

In [ ]:
yeni_kelimeler = ["espresso", "minimal", "gürültü", "sessizlik", "poster"]
for kelime in yeni_kelimeler:
    vektorler_sozlugu[kelime] = vektor_al(kelime)

print(len(vektorler_sozlugu))